In [1]:
!pip install apache-airflow apache-airflow-providers-apache-spark pyspark requests

In [2]:

from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.providers.apache.spark.operators.spark_submit import SparkSubmitOperator
from datetime import datetime, timedelta
import requests

default_args = {
    "owner": "your-group",
    "retries": 2,
    "retry_delay": timedelta(minutes=1),
}

def check_connector():
    r = requests.get("http://connect:8083/connectors/cdc-connector/status")
    # Add a check for successful response code before parsing JSON
    if r.status_code != 200:
        raise ValueError(f"Failed to reach connector API: Status {r.status_code}")
    
    status = r.json()["connector"]["state"]
    assert status == "RUNNING", f"Connector is {status}, not RUNNING"

with DAG(
    dag_id="project3_pipeline",
    default_args=default_args,
    start_date=datetime(2026, 4, 1),
    schedule="*/15 * * * *",   # every 15 minutes
    catchup=False,
    tags=["project3", "lakehouse", "iceberg"],
) as dag:

    # --- TASK 1: CDC Path ---
    health_check = PythonOperator(
        task_id="check_debezium_health",
        python_callable=check_connector,
    )

    bronze_cdc = SparkSubmitOperator(
        task_id="bronze_cdc",
        application="/path/to/your/bronze_cdc_script.py", # Update path
        conn_id="spark_default"
    )

    silver_cdc = SparkSubmitOperator(
        task_id="silver_cdc",
        application="/path/to/your/silver_cdc_script.py", # Update path
        conn_id="spark_default"
    )

    # --- TASK 2: Taxi Path ---
    bronze_taxi = SparkSubmitOperator(
        task_id="bronze_taxi",
        application="/path/to/your/bronze_taxi_script.py", # Update path
        conn_id="spark_default"
    )

    silver_taxi = SparkSubmitOperator(
        task_id="silver_taxi",
        application="/path/to/your/silver_taxi_script.py", # Update path
        conn_id="spark_default"
    )

    # --- Wiring the DAG dependencies ---
    # CDC path — needs Debezium to be healthy
    health_check >> bronze_cdc >> silver_cdc

    # Taxi path — independent, reads parquet files
    bronze_taxi >> silver_taxi

2026-04-30T12:50:10.525400Z [warning  ] The `airflow.operators.python.PythonOperator` attribute is deprecated. Please use `'airflow.providers.standard.operators.python.PythonOperator'`. [py.warnings] category=DeprecatedImportWarning filename=/tmp/ipykernel_1026/2749991382.py lineno=2


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import monotonically_increasing_id

spark = SparkSession.builder.appName("TaxiBronzeIngestion").getOrCreate()

# Create namespace
spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.taxi")

# Read raw parquet from data/
df_raw = spark.read.parquet("data/taxi_data_*.parquet") # Adjust path to your data

# Add synthetic trip_id
df_bronze = df_raw.withColumn("trip_id", monotonically_increasing_id())

# Write to Bronze Iceberg table
df_bronze.write.format("iceberg") \
    .mode("append") \
    .saveAsTable("lakehouse.taxi.bronze_trips")

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("TaxiSilverRefinement").getOrCreate()

df_bronze = spark.read.table("lakehouse.taxi.bronze_trips")

# Clean types, handle nulls, filter invalid rows
df_silver = df_bronze.filter(
    (col("fare_amount") > 0) & 
    (col("trip_distance") > 0)
).dropna(subset=["PULocationID", "tpep_pickup_datetime"]) # Adjust to necessary not-null columns

# Write to Silver Iceberg table
df_silver.write.format("iceberg") \
    .mode("append") \
    .saveAsTable("lakehouse.taxi.silver_trips")

In [ ]:
-- gold_prototype.sql
-- Prototyping the Gold table join logic

WITH customer_count AS (
    SELECT count(*) AS cnt FROM lakehouse.cdc.silver_customers
)
SELECT
    t.PULocationID,
    t.fare_amount,
    t.trip_distance,
    t.tpep_pickup_datetime,
    c.name AS customer_name,
    c.country AS customer_country
FROM lakehouse.taxi.silver_trips t
CROSS JOIN customer_count cc
JOIN lakehouse.cdc.silver_customers c
    ON MOD(t.trip_id, cc.cnt) + 1 = c.id
LIMIT 20;

In [ ]:
print("ok")